# Yahoo Finance–Only Investment Risk Pipeline (PySpark + Delta Lake)

This notebook downloads historical daily bars for a configured universe of stocks, ETFs and benchmark indices from Yahoo Finance using `yfinance`. It standardizes the data, validates it, writes Bronze/Silver/Gold Delta tables, and calculates initial risk features.

**Important:** `yfinance` is an unofficial interface. Yahoo Finance may throttle requests, return incomplete history, or no longer provide data for delisted/renamed tickers. Yahoo Finance does not provide a reliable complete ticker directory through this interface, so configure your own universe or supply a CSV. Start with a small smoke test.

## 1. Install dependencies

Recommended local environment: Python 3.11, Java 17, VS Code with Python and Jupyter extensions. Use the matching Spark/Delta versions below for a local Spark setup.

In [ ]:
%pip install -q pyspark==3.5.6 delta-spark==3.3.2 yfinance==0.2.66 pandas pyarrow

In [ ]:
import time, logging
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import yfinance as yf

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("yahoo-market-pipeline")

## 2. Configure paths and ticker universe

You can edit `DEFAULT_UNIVERSE` or provide a CSV with columns `symbol,asset_type`. Valid asset types are `STOCK`, `ETF`, and `INDEX`.

The sample list is deliberately small. Set `MAX_TICKERS=None` after validating the pipeline.

In [ ]:
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data/raw/yahoo_finance"
BRONZE_PATH = PROJECT_ROOT / "data/lakehouse/bronze/market_bars"
SILVER_PATH = PROJECT_ROOT / "data/lakehouse/silver/market_bars"
GOLD_PATH = PROJECT_ROOT / "data/lakehouse/gold/investment_risk_features"
QUARANTINE_PATH = PROJECT_ROOT / "data/lakehouse/quarantine/market_bars"
REPORT_DIR = PROJECT_ROOT / "data/reports"

for p in [RAW_DIR, BRONZE_PATH.parent, SILVER_PATH.parent,
          GOLD_PATH.parent, QUARANTINE_PATH.parent, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

UNIVERSE_CSV = None  # e.g. PROJECT_ROOT / "config/ticker_universe.csv"

DEFAULT_UNIVERSE = [
    ("AAPL", "STOCK"), ("MSFT", "STOCK"), ("NVDA", "STOCK"),
    ("AMZN", "STOCK"), ("GOOGL", "STOCK"), ("META", "STOCK"),
    ("JPM", "STOCK"), ("TSLA", "STOCK"),
    ("SPY", "ETF"), ("QQQ", "ETF"), ("IWM", "ETF"),
    ("GLD", "ETF"), ("TLT", "ETF"),
    ("^GSPC", "INDEX"), ("^IXIC", "INDEX"),
    ("^DJI", "INDEX"), ("^RUT", "INDEX")
]

MAX_TICKERS = 20       # None = all configured tickers
START_DATE = None      # None = maximum available history
END_DATE = None        # None = latest available history
INTERVAL = "1d"
AUTO_ADJUST = True
BATCH_SIZE = 10
MAX_RETRIES = 3
RETRY_BASE_SECONDS = 3
BATCH_PAUSE_SECONDS = 1.5

def load_universe():
    if UNIVERSE_CSV is not None:
        df = pd.read_csv(UNIVERSE_CSV)
        if not {"symbol", "asset_type"}.issubset(df.columns):
            raise ValueError("Universe CSV must contain symbol and asset_type columns")
        df = df[["symbol", "asset_type"]].dropna()
    else:
        df = pd.DataFrame(DEFAULT_UNIVERSE, columns=["symbol", "asset_type"])
    df["symbol"] = df["symbol"].astype(str).str.strip().str.upper()
    df["asset_type"] = df["asset_type"].astype(str).str.strip().str.upper()
    if not set(df.asset_type).issubset({"STOCK", "ETF", "INDEX"}):
        raise ValueError("asset_type must be STOCK, ETF or INDEX")
    df = df[df.symbol != ""].drop_duplicates("symbol").reset_index(drop=True)
    return df.head(MAX_TICKERS).copy() if MAX_TICKERS is not None else df

universe = load_universe()
display(universe)
print("Configured tickers:", len(universe))

## 3. Start Spark with Delta Lake

In [ ]:
builder = (
    SparkSession.builder
    .appName("YahooFinance-InvestmentRisk")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

## 4. Download historical daily bars

Downloads are batched and retried with exponential backoff. Each successful batch is saved as Parquet immediately, avoiding the need to hold the entire universe in memory.

The normalizer handles the MultiIndex columns commonly returned by multi-ticker `yfinance.download()` calls.

In [ ]:
def batches(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]

def normalize_download(data, ticker):
    if data is None or data.empty:
        return pd.DataFrame()
    df = data.copy()
    if isinstance(df.columns, pd.MultiIndex):
        level0 = set(map(str, df.columns.get_level_values(0)))
        level1 = set(map(str, df.columns.get_level_values(1)))
        if ticker in level0:
            df = df[ticker]
        elif ticker in level1:
            df = df.xs(ticker, axis=1, level=1)
        else:
            return pd.DataFrame()
    df.columns = [str(c).strip() for c in df.columns]
    needed = {"Open", "High", "Low", "Close", "Volume"}
    if not needed.issubset(df.columns):
        return pd.DataFrame()
    df = df.reset_index()
    date_col = "Date" if "Date" in df.columns else ("Datetime" if "Datetime" in df.columns else df.columns[0])
    dates = pd.to_datetime(df[date_col], errors="coerce", utc=True).dt.tz_convert(None)
    out = pd.DataFrame({
        "symbol": ticker,
        "observation_date": dates,
        "open": pd.to_numeric(df["Open"], errors="coerce"),
        "high": pd.to_numeric(df["High"], errors="coerce"),
        "low": pd.to_numeric(df["Low"], errors="coerce"),
        "close": pd.to_numeric(df["Close"], errors="coerce"),
        "volume": pd.to_numeric(df["Volume"], errors="coerce")
    })
    # auto_adjust=True adjusts OHLC. adjusted_close mirrors close; do not adjust twice.
    out["adjusted_close"] = out["close"]
    out["currency"] = None
    out["source"] = "YAHOO_FINANCE_YFINANCE"
    out["price_basis"] = "AUTO_ADJUSTED_OHLC" if AUTO_ADJUST else "UNADJUSTED_OHLC"
    out["ingestion_timestamp"] = datetime.now(timezone.utc).replace(tzinfo=None)
    return out.dropna(subset=["observation_date"])

def download_with_retry(tickers):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            data = yf.download(
                tickers=tickers, start=START_DATE, end=END_DATE,
                period="max" if START_DATE is None and END_DATE is None else None,
                interval=INTERVAL, auto_adjust=AUTO_ADJUST,
                actions=False, group_by="ticker", threads=True,
                progress=False, timeout=30, multi_level_index=True
            )
            return data, None
        except Exception as exc:
            last_error = repr(exc)
            wait = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            logger.warning("Attempt %d failed: %s; waiting %ss", attempt, last_error, wait)
            if attempt < MAX_RETRIES:
                time.sleep(wait)
    return pd.DataFrame(), last_error

for f in RAW_DIR.glob("batch_*.parquet"):
    f.unlink()

success_records, failure_records, batch_records = [], [], []
ticker_list = universe.symbol.tolist()

for batch_no, ticker_batch in enumerate(batches(ticker_list, BATCH_SIZE), start=1):
    logger.info("Downloading batch %d: %s", batch_no, ticker_batch)
    data, batch_error = download_with_retry(ticker_batch)
    frames = []
    for ticker in ticker_batch:
        one = normalize_download(data, ticker)
        if one.empty:
            failure_records.append({
                "symbol": ticker,
                "reason": batch_error or "No usable history returned"
            })
        else:
            frames.append(one)
            success_records.append({
                "symbol": ticker,
                "rows": len(one),
                "first_date": str(one.observation_date.min().date()),
                "last_date": str(one.observation_date.max().date())
            })
    if frames:
        combined = pd.concat(frames, ignore_index=True)
        combined.to_parquet(RAW_DIR / f"batch_{batch_no:05d}.parquet", index=False)
        batch_records.append({
            "batch": batch_no, "tickers_requested": len(ticker_batch),
            "tickers_with_data": len(frames), "rows": len(combined)
        })
    time.sleep(BATCH_PAUSE_SECONDS)

success_df = pd.DataFrame(success_records)
failure_df = pd.DataFrame(failure_records)
batch_df = pd.DataFrame(batch_records)

print("Tickers with data:", success_df.symbol.nunique() if not success_df.empty else 0)
print("Tickers failed:", failure_df.symbol.nunique() if not failure_df.empty else 0)
display(success_df.head(10))
display(failure_df)

## 5. Save download audit reports

In [ ]:
success_df.to_csv(REPORT_DIR / "download_success.csv", index=False)
failure_df.to_csv(REPORT_DIR / "download_failures.csv", index=False)
batch_df.to_csv(REPORT_DIR / "download_batches.csv", index=False)
print("Reports:", REPORT_DIR.resolve())

## 6. Load downloaded Parquet into Spark

The asset type is attached from the configured universe. The canonical table grain is one row per (`symbol`, `observation_date`).

In [ ]:
parquet_paths = [str(p) for p in sorted(RAW_DIR.glob("batch_*.parquet"))]
if not parquet_paths:
    raise RuntimeError("No data downloaded. Check download_failures.csv, ticker symbols and network access.")

downloaded = spark.read.parquet(*parquet_paths)
universe_schema = T.StructType([
    T.StructField("symbol", T.StringType(), False),
    T.StructField("asset_type", T.StringType(), False)
])
universe_spark = spark.createDataFrame(
    [(r.symbol, r.asset_type) for r in universe.itertuples(index=False)],
    schema=universe_schema
)

bars = (
    downloaded.join(universe_spark, "symbol", "inner")
    .select(
        "symbol", "asset_type", "observation_date",
        "open", "high", "low", "close", "adjusted_close",
        "volume", "currency", "source", "price_basis", "ingestion_timestamp"
    )
    .withColumn("symbol", F.upper(F.trim("symbol")))
    .withColumn("observation_date", F.to_date("observation_date"))
)
print("Downloaded rows:", bars.count())
bars.printSchema()

## 7. Validate and deduplicate records

Checks cover missing keys, non-positive closing prices, negative volume, impossible OHLC relationships and duplicate symbol/date keys. Null volume is allowed (some index feeds do not provide it).

In [ ]:
duplicate_keys = (
    bars.groupBy("symbol", "observation_date").count()
    .filter(F.col("count") > 1)
)

dedupe_w = Window.partitionBy("symbol", "observation_date").orderBy(
    F.col("ingestion_timestamp").desc_nulls_last()
)
deduped = (
    bars.withColumn("_rn", F.row_number().over(dedupe_w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

invalid = (
    F.col("symbol").isNull()
    | F.col("observation_date").isNull()
    | F.col("close").isNull()
    | (F.col("close") <= 0)
    | (F.col("volume") < 0)
    | (F.col("high") < F.col("low"))
    | (F.col("high") < F.greatest("open", "close"))
    | (F.col("low") > F.least("open", "close"))
)

quarantine = deduped.filter(invalid)
silver = deduped.filter(~invalid)

quality_metrics = silver.agg(
    F.count("*").alias("valid_rows"),
    F.countDistinct("symbol").alias("unique_symbols"),
    F.min("observation_date").alias("first_date"),
    F.max("observation_date").alias("last_date"),
    F.sum(F.col("close").isNull().cast("long")).alias("null_close"),
    F.sum((F.col("close") <= 0).cast("long")).alias("nonpositive_close"),
    F.sum((F.col("volume") < 0).cast("long")).alias("negative_volume")
).first().asDict()

print("Duplicate keys:", duplicate_keys.count())
print("Quarantined rows:", quarantine.count())
print("Quality metrics:", quality_metrics)
display(duplicate_keys.orderBy(F.desc("count")).limit(20))
display(silver.groupBy("asset_type").agg(
    F.count("*").alias("rows"),
    F.countDistinct("symbol").alias("symbols"),
    F.min("observation_date").alias("first_date"),
    F.max("observation_date").alias("last_date")
))

## 8. Write Bronze, Silver and Quarantine Delta tables

- Bronze: standardized, deduplicated bars before validation filtering.
- Silver: valid canonical market bars.
- Quarantine: invalid records retained for inspection.

This is an initial full build and uses `overwrite`. For recurring ingestion, implement Delta `MERGE` keyed by (`symbol`, `observation_date`) rather than overwriting the full history.

In [ ]:
def write_delta(df, path):
    (df.withColumn("year", F.year("observation_date"))
       .withColumn("month", F.month("observation_date"))
       .write.format("delta").mode("overwrite")
       .partitionBy("year", "month").save(str(path)))

write_delta(deduped, BRONZE_PATH)
write_delta(silver, SILVER_PATH)
if quarantine.limit(1).count() > 0:
    write_delta(quarantine, QUARANTINE_PATH)

print("Bronze:", BRONZE_PATH)
print("Silver:", SILVER_PATH)
print("Quarantine:", QUARANTINE_PATH)

## 9. Calculate returns and benchmark features

Returns are calculated independently per ticker. Index returns are pivoted by date and joined to stock/ETF rows. Missing benchmark returns remain null; they are never replaced with zero.

In [ ]:
asset_w = Window.partitionBy("symbol").orderBy("observation_date")
with_returns = (
    silver
    .withColumn("previous_close", F.lag("close").over(asset_w))
    .withColumn(
        "daily_return",
        F.when(
            F.col("previous_close").isNotNull() & (F.col("previous_close") > 0),
            F.col("close") / F.col("previous_close") - 1.0
        )
    )
)

index_symbols = universe.loc[universe.asset_type == "INDEX", "symbol"].tolist()
index_returns = (
    with_returns.filter(F.col("asset_type") == "INDEX")
    .select("observation_date", "symbol", F.col("daily_return").alias("benchmark_return"))
)

stock_etf = with_returns.filter(F.col("asset_type").isin("STOCK", "ETF"))
if index_symbols:
    benchmark_wide = (
        index_returns.groupBy("observation_date")
        .pivot("symbol", index_symbols)
        .agg(F.first("benchmark_return", ignorenulls=True))
    )
    gold = stock_etf.join(benchmark_wide, "observation_date", "left")
else:
    gold = stock_etf

## 10. Create rolling risk features and write Gold Delta

In [ ]:
w20 = Window.partitionBy("symbol").orderBy("observation_date").rowsBetween(-19, 0)
w60 = Window.partitionBy("symbol").orderBy("observation_date").rowsBetween(-59, 0)

gold = (
    gold
    .withColumn("volatility_20d", F.stddev_samp("daily_return").over(w20) * F.sqrt(F.lit(252.0)))
    .withColumn("volatility_60d", F.stddev_samp("daily_return").over(w60) * F.sqrt(F.lit(252.0)))
    .withColumn("avg_volume_20d", F.avg("volume").over(w20))
    .withColumn("rolling_max_20d", F.max("close").over(w20))
    .withColumn("rolling_max_60d", F.max("close").over(w60))
    .withColumn("drawdown_20d", F.col("close") / F.col("rolling_max_20d") - 1.0)
    .withColumn("drawdown_60d", F.col("close") / F.col("rolling_max_60d") - 1.0)
)

write_delta(gold, GOLD_PATH)
print("Gold Delta:", GOLD_PATH)

## 11. Inspect Gold dataset

In [ ]:
gold_read = spark.read.format("delta").load(str(GOLD_PATH))
print("Gold rows:", gold_read.count())
print("Gold assets:", gold_read.select("symbol").distinct().count())

gold_read.agg(
    F.min("observation_date").alias("first_date"),
    F.max("observation_date").alias("last_date")
).show()

display(gold_read.groupBy("asset_type").agg(
    F.count("*").alias("rows"),
    F.countDistinct("symbol").alias("symbols"),
    F.min("observation_date").alias("first_date"),
    F.max("observation_date").alias("last_date")
))

display(gold_read.select(
    "symbol", "observation_date", "close", "daily_return",
    "volatility_20d", "volatility_60d", "drawdown_20d"
).orderBy("symbol", "observation_date").limit(30))

## 12. Export coverage and quality reports

In [ ]:
configured = set(universe.symbol)
received = set(success_df.symbol) if not success_df.empty else set()
failed = set(failure_df.symbol) if not failure_df.empty else set()

coverage = pd.DataFrame([{
    "configured_tickers": len(configured),
    "tickers_with_data": len(received),
    "tickers_without_data": len(failed),
    "coverage_pct": round(100 * len(received) / max(1, len(configured)), 2),
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat()
}])
coverage.to_csv(REPORT_DIR / "universe_coverage.csv", index=False)
pd.DataFrame([quality_metrics]).to_csv(REPORT_DIR / "silver_quality_metrics.csv", index=False)
display(coverage)
print("Reports saved to:", REPORT_DIR.resolve())

## 13. Important considerations before modelling

1. Review failed tickers and rerun them later if failures appear temporary.
2. Different IPO dates naturally produce different history lengths.
3. Ticker symbols can change or be reused. For robust research, maintain stable security identifiers and effective dates.
4. A current ticker universe introduces survivorship bias into historical backtests. Delisted securities may be missing.
5. Use chronological or walk-forward validation, not random row splits.
6. This notebook uses adjusted OHLC when `AUTO_ADJUST=True`. Keep this price basis consistent throughout downstream analytics.
7. Check Yahoo Finance's applicable terms and data-use permissions before commercial use or redistribution.

### Next milestones
- Incremental ingestion with Delta `MERGE`.
- Portfolio holdings/weights table and portfolio return calculation.
- Beta, Sharpe ratio, correlation, historical/parametric VaR and CVaR.
- Walk-forward volatility forecasting and risk-regime classification.
- Kafka + Spark Structured Streaming, MLflow, Airflow and monitoring.

**Outputs:** Yahoo Finance-only Bronze, Silver and Gold Delta datasets. No Kaggle legacy data or Nasdaq snapshot dependency.